# Solar AI — Ground-level PV defect research detector (YOLO11n)

Trains `ground_yolo11n_baseline_v1` (4 classes: `panel / bird_drop / cracked / dusty`) on a
Kaggle GPU. **Separate research model** — does not touch the production aerial detector
`weights/yolo_solar.pt`.

- Base model: pretrained **YOLO11n** COCO checkpoint (never the aerial Solar AI model).
- Dataset: Kaggle Dataset built from the audited, class-normalized Roboflow *panel solar* v1
  (`susan-ifblr/panel-solar-bw945`, CC BY 4.0 as displayed by the source).
- imgsz=640, epochs=100, patience=20, seed=42.

Attach the dataset in the notebook sidebar as **`<your-kaggle-username>/solar-ai-ground-pv-research-v1`**
before running. Set `KAGGLE_DATASET_SLUG` below to match.

## 1. Environment

In [ ]:
import sys, platform, subprocess, json, os, time, hashlib, shutil
from pathlib import Path
print('Python :', platform.python_version())
try:
    import torch, torchvision
    print('torch  :', torch.__version__)
    print('tv     :', torchvision.__version__)
except Exception as e:
    print('torch not yet importable:', e)

### CUDA — fail closed if unavailable
Kaggle's free GPU pool can hand out a Pascal **Tesla P100 (sm_60)**. Newer torch cu12.8
builds dropped sm_60 kernels, so we pin `torch==2.7.1` / `torchvision==0.22.1` (last line
retaining Pascal support, CUDA 12.6 wheels) — same pins proven on this account's prior runs.

In [ ]:
PINNED = ['torch==2.7.1', 'torchvision==0.22.1', 'ultralytics==8.4.138', 'PyYAML>=6.0.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED], check=True)

import torch
assert torch.cuda.is_available(), 'CUDA is not available — this notebook requires a Kaggle GPU (Settings -> Accelerator -> GPU T4/P100).'
print('torch      :', torch.__version__)
print('CUDA       :', torch.version.cuda)
print('device     :', torch.cuda.get_device_name(0))
print('capability :', torch.cuda.get_device_capability(0))
props = torch.cuda.get_device_properties(0)
print('memory GB  :', round(props.total_memory / 1e9, 1))
import ultralytics; print('ultralytics:', ultralytics.__version__)

## 2. Locate the attached Kaggle dataset

In [ ]:
KAGGLE_DATASET_SLUG = os.environ.get('KAGGLE_DATASET_SLUG', 'solar-ai-ground-pv-research-v1')

# Kaggle mounts datasets under /kaggle/input/<slug>/ OR /kaggle/input/datasets/<owner>/<slug>/
candidates = list(Path('/kaggle/input').rglob('data.yaml'))
assert candidates, 'No data.yaml found under /kaggle/input — attach the dataset in the sidebar first.'
DATA_YAML = None
for c in candidates:
    if KAGGLE_DATASET_SLUG in str(c) or 'ground' in str(c).lower() or len(candidates) == 1:
        DATA_YAML = c; break
DATA_YAML = DATA_YAML or candidates[0]
DATA_ROOT = DATA_YAML.parent
print('data.yaml :', DATA_YAML)
print(DATA_YAML.read_text())

## 3. Validate data.yaml + dataset statistics (no training yet)

In [ ]:
import yaml
cfg = yaml.safe_load(DATA_YAML.read_text())
names = cfg['names']
if isinstance(names, list): names = {i: n for i, n in enumerate(names)}
names = {int(k): v for k, v in names.items()}
EXPECTED = {0: 'panel', 1: 'bird_drop', 2: 'cracked', 3: 'dusty'}
assert names == EXPECTED, f'class taxonomy mismatch: {names} != {EXPECTED}'
print('class taxonomy OK:', names)

from collections import Counter
for split_key in ('train', 'val', 'test'):
    rel = cfg.get(split_key)
    if not rel: continue
    img_dir = (DATA_ROOT / rel)
    lbl_dir = Path(str(img_dir).replace('/images', '/labels'))
    imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    inst, img_with = Counter(), Counter()
    for im in imgs:
        lp = lbl_dir / (im.stem + '.txt')
        if not lp.is_file(): continue
        seen = set()
        for line in lp.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                cid = int(p[0]); inst[EXPECTED[cid]] += 1; seen.add(EXPECTED[cid])
        for s in seen: img_with[s] += 1
    print(f'  {split_key:5s}  images={len(imgs):5d}  instances={dict(inst)}  images_with={dict(img_with)}')

## 4. Train `ground_yolo11n_baseline_v1`
Base model: `yolo11n.pt` (COCO-pretrained). Resolved training args are printed before the run.

In [ ]:
from ultralytics import YOLO

PROJECT = '/kaggle/working/research_runs/ground_pv'
RUN_NAME = 'ground_yolo11n_baseline_v1'
BASE_MODEL = 'yolo11n.pt'

train_args = dict(
    data=str(DATA_YAML), epochs=100, imgsz=640, patience=20, seed=42,
    batch=-1, device=0, project=PROJECT, name=RUN_NAME, exist_ok=True,
    pretrained=True, cos_lr=True, optimizer='auto', verbose=True,
    # conservative augmentation (must not distort defect semantics)
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, degrees=5.0, translate=0.1, scale=0.3,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=0.5, mixup=0.0, copy_paste=0.0,
)
print(json.dumps({k: str(v) for k, v in train_args.items()}, indent=2))

t0 = time.time()
model = YOLO(BASE_MODEL)
results = model.train(**train_args)
train_seconds = time.time() - t0
RUN_DIR = Path(results.save_dir)
print('run dir:', RUN_DIR, ' elapsed(s):', round(train_seconds))

## 5. Evaluate — validation and test, with per-class metrics

In [ ]:
best = RUN_DIR / 'weights' / 'best.pt'
assert best.is_file(), best
eval_model = YOLO(str(best))

def evaluate(split):
    m = eval_model.val(data=str(DATA_YAML), split=split, imgsz=640, iou=0.5, plots=True, verbose=False)
    box = m.box
    per_class = {}
    for pos, cidx in enumerate(list(box.ap_class_index)):
        p, r, ap50, ap = box.class_result(pos)
        per_class[EXPECTED[int(cidx)]] = dict(precision=float(p), recall=float(r), mAP50=float(ap50), mAP50_95=float(ap))
    return dict(
        overall=dict(precision=float(box.mp), recall=float(box.mr), mAP50=float(box.map50), mAP50_95=float(box.map)),
        per_class=per_class, save_dir=str(m.save_dir))

evaluation = {}
for split in ('val', 'test'):
    try:
        evaluation[split] = evaluate(split)
        print(split, json.dumps(evaluation[split]['overall'], indent=2))
        print('  per-class:', json.dumps(evaluation[split]['per_class'], indent=2))
    except Exception as e:
        print(f'{split}: skipped ({e})')

## 6. Package artifacts + manifest + SHA-256

In [ ]:
OUT = Path('/kaggle/working/output'); OUT.mkdir(exist_ok=True)
for name in ('best.pt', 'last.pt'):
    src = RUN_DIR / 'weights' / name
    if src.is_file(): shutil.copy2(src, OUT / name)
for pat in ('results.csv','results.png','confusion_matrix.png','confusion_matrix_normalized.png',
            'BoxPR_curve.png','BoxF1_curve.png','BoxP_curve.png','BoxR_curve.png',
            'PR_curve.png','F1_curve.png','P_curve.png','R_curve.png'):
    for f in RUN_DIR.rglob(pat): shutil.copy2(f, OUT / f.name)
pred_dir = OUT / 'predictions'; pred_dir.mkdir(exist_ok=True)
val_imgs = sorted((DATA_ROOT / cfg['val']).glob('*'))[:12]
if val_imgs:
    for r in YOLO(str(best)).predict(source=[str(p) for p in val_imgs], conf=0.25, save=True, project=str(pred_dir), name='val_examples', exist_ok=True):
        pass

sha = hashlib.sha256((OUT / 'best.pt').read_bytes()).hexdigest()
git_sha = None
try: git_sha = subprocess.check_output(['git','-C','/kaggle/working/solar-ai-framework','rev-parse','HEAD'], text=True).strip()
except Exception: pass

manifest = dict(
    experiment_name=RUN_NAME,
    dataset_name='panel solar', dataset_version=1,
    dataset_source='https://universe.roboflow.com/susan-ifblr/panel-solar-bw945',
    dataset_license='CC BY 4.0 (as displayed by Roboflow)',
    model=BASE_MODEL, pretrained_source='YOLO11n COCO-pretrained (Ultralytics)',
    epochs=train_args['epochs'], imgsz=train_args['imgsz'], batch=train_args['batch'],
    optimizer='auto', seed=42, device='cuda:0', gpu=torch.cuda.get_device_name(0),
    python_version=platform.python_version(), torch_version=torch.__version__,
    ultralytics_version=ultralytics.__version__,
    train_seconds=train_seconds, run_dir=str(RUN_DIR),
    best_metrics=evaluation.get('val', {}).get('overall'),
    evaluation=evaluation, checkpoint_sha256=sha, git_commit_if_available=git_sha,
)
(OUT / 'training_manifest.json').write_text(json.dumps(manifest, indent=2))
(OUT / 'evaluation.json').write_text(json.dumps(evaluation, indent=2))
print('best.pt sha256:', sha)
print(json.dumps(manifest, indent=2, default=str))
print('\nArtifacts:', sorted(p.name for p in OUT.iterdir()))

## 7. Retrieve
Download `/kaggle/working/output/` (or add it as a new **Kaggle Dataset version**).
Locally, save `best.pt` as `weights/yolo_solar_ground_research_v1.pt` — **never** overwrite
`weights/yolo_solar.pt` — then run `scripts/evaluate_ground_pv.py` and `scripts/infer_ground_pv.py`.
Record the SHA-256 in `datasets/ground_pv_research/reports/ground_research_model_manifest.json`.